## This notebook allows quick analysis of the results from the CLI friedel pair finder

#### Set static paths, keys and parameters

In [ ]:
import tomllib
from pathlib import Path

# ── Change these two lines ───────────────────────────────────────────────────
OUT_PATH = "output/Al_big_grains_logtif.h5"
DATASET  = "Al_big_grains"
CONTEXT  = 8                    # padding pixels around each spot bbox

# ── Derived from registry ────────────────────────────────────────────────────
with open("configs/scan_registry.toml", "rb") as _f:
    _reg = tomllib.load(_f)

_cfg      = _reg["scans"][DATASET]
_keys     = _reg["hdf5_keys"]
DATA_ROOT = Path("DATA/ESRF_May_2026")  # adjust if running outside devcontainer
_scan_dir = DATA_ROOT / DATASET

RAW_PATH = str(_scan_dir / _cfg["raw"])
RAW_KEY  = _keys["raw"]

if "log" in _cfg:
    SEG_PATH = str(_scan_dir / _cfg["log"])
    SEG_TYPE = "log_tif"
else:
    SEG_PATH = str(_scan_dir / _cfg["seg"])
    SEG_TYPE = "seg_vol"
SEG_KEY = _keys["seg"]

#### Import packages

In [ ]:
import h5py
import hdf5plugin  # registers bitshuffle/blosc codecs for ESRF HDF5 files
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

#### Load spots from result file

In [ ]:
def _s(v): return v.decode() if isinstance(v, bytes) else str(v)

spots = {}
BEAM_CENTER_Y = BEAM_CENTER_X = BEAM_HALF_HEIGHT = BEAM_HALF_WIDTH = None
_config_raw: dict = {}
with h5py.File(OUT_PATH, "r") as f:
    scan_name = next(iter(f.keys()))
    grp = f[scan_name]
    if "config" in grp:
        cfg = grp["config"]
        _config_raw = {k: cfg[k][()] for k in cfg.keys()}
        _stored_seg_path = _s(cfg["seg_path"][()])
        if Path(_stored_seg_path).exists():
            SEG_PATH = _stored_seg_path
            SEG_TYPE = _s(cfg["seg_type"][()])
            SEG_KEY  = _s(cfg["seg_key"][()])
            print("Config loaded from result file.")
        else:
            print(f"Stored seg_path not found on this machine ({_stored_seg_path}).")
            print("Using fallback SEG_PATH/SEG_TYPE/SEG_KEY from the config cell above.")
        BEAM_CENTER_Y    = float(cfg["beam_center_y"][()]) if "beam_center_y" in cfg else None
        BEAM_CENTER_X    = float(cfg["beam_center_x"][()]) if "beam_center_x" in cfg else None
        BEAM_HALF_HEIGHT = float(cfg["beam_half_height"][()]) if "beam_half_height" in cfg else None
        BEAM_HALF_WIDTH  = float(cfg["beam_half_width"][()]) if "beam_half_width" in cfg else None
    else:
        print("No config group in result file — using fallback values from config cell.")

    sg          = grp["spots"]
    blob_ids    = sg["blob_id"][()]
    frame_idxs  = sg["frame_idx"][()]
    partner_ids = sg["friedel_partner_id"][()]
    ncc_scores  = sg["ncc_score"][()]
    centroids   = sg["centroid"][()]
    shapes      = sg["shape"][()]
    offsets     = sg["offset"][()]
    masks_flat  = sg["masks"][()]
    for i, bid in enumerate(blob_ids):
        h, w  = int(shapes[i, 0]), int(shapes[i, 1])
        start = int(offsets[i])
        mask  = masks_flat[start : start + h * w].reshape(h, w).astype(bool)
        spots[int(bid)] = dict(
            mask      = mask,
            centroid  = (float(centroids[i, 0]), float(centroids[i, 1])),
            frame_idx = int(frame_idxs[i]),
            partner   = int(partner_ids[i]),
            ncc       = float(ncc_scores[i]),
        )

print(f"Scan:             {scan_name}")
print(f"Seg path:         {SEG_PATH}  [{SEG_TYPE}]")
print(f"Beam center y:    {BEAM_CENTER_Y if BEAM_CENTER_Y is not None else 'not stored'}")
print(f"Beam center x:    {BEAM_CENTER_X if BEAM_CENTER_X is not None else 'not stored'}")
print(f"Beam half height: {BEAM_HALF_HEIGHT if BEAM_HALF_HEIGHT is not None else 'not stored'}")
print(f"Beam half width:  {BEAM_HALF_WIDTH if BEAM_HALF_WIDTH is not None else 'not stored'}")

# -- Get unique pairs sorted by frame ------------------------------------------
pairs = [(s, spots[pid]) for bid, s in spots.items()
         if (pid := s["partner"]) in spots and bid < pid]
pairs.sort(key=lambda p: p[0]["frame_idx"])
print(f"{len(pairs)} unique Friedel pairs")

# -- Seg frame loader (LoG TIF or seg_vol) with cache --------------------------
_frame_cache: dict = {}

def _load_frame(frame_idx: int):
    if frame_idx in _frame_cache:
        return _frame_cache[frame_idx]
    try:
        if SEG_TYPE == "seg_vol":
            with h5py.File(SEG_PATH, "r") as f:
                arr = f[SEG_KEY][frame_idx].astype(np.float32)
            arr = np.rot90(arr[::-1], k=-1)
        else:
            p = Path(SEG_PATH) / f"proj{frame_idx:04d}.tif"
            arr = np.array(Image.open(p), dtype=np.float32)
    except Exception as e:
        print(f"Could not load seg frame {frame_idx}: {e}")
        return None
    _frame_cache[frame_idx] = arr
    return arr

# -- Raw frame loader with cache -----------------------------------------------
_raw_cache: dict = {}

def _load_raw_frame(frame_idx: int):
    if frame_idx in _raw_cache:
        return _raw_cache[frame_idx]
    try:
        with h5py.File(RAW_PATH, "r") as f:
            arr = f[RAW_KEY][frame_idx].astype(np.float32)
    except Exception as e:
        print(f"Could not load raw frame {frame_idx}: {e}")
        return None
    _raw_cache[frame_idx] = arr
    return arr

### Print result files meta information

In [ ]:
_PARAM_DESC = {
    "seg_path":          "Path to the segmentation source used as pipeline input",
    "seg_key":           "HDF5 dataset key within the segmentation file",
    "seg_type":          "Segmentation format: seg_vol (uint8 HDF5) or log_tif (float32 LoG TIFs)",
    "n_frames":          "Total valid rotation frames in the scan",
    "frames":            "Frame range processed (rest skipped, good for partial runs)",
    "frame_height":      "Frame height in pixels",
    "beam_center_y":     "Estimated y-coordinate of the direct beam centre (px)",
    "beam_center_x":     "Estimated x-coordinate of the direct beam centre (px)",
    "beam_half_height":  "Half-height of the illuminated beam slab; sets Y midpoint search window (px)",
    "beam_half_width":   "Half-width of the illuminated beam; sets X midpoint search window (px)",
    "seg_vol_threshold": "Binarization threshold for seg_vol frames (uint8 scale)",
    "log_threshold":     "Binarization threshold for LoG TIF frames (float32 scale)",
    "min_blob_area":     "Minimum connected-component area kept after Pass 1 (px)",
    "max_blob_area":     "Maximum connected-component area kept after Pass 1 (px)",
    "omega_tolerance":   "Frame search window around expected Friedel offset (\u00b1frames)",
    "min_area_ratio":    "Minimum area(small)/area(large) ratio required between Friedel partners",
    "min_ncc":           "Minimum Pearson NCC to accept a Friedel pair",
    "ncc_area_weight":   "Tightens NCC threshold for small blobs: +k/\u221aarea  (0 = disabled)",
}

if _config_raw:
    print(f"{'Parameter':<24} {'Value':<15} Description")
    print("\u2500" * 100)
    for key in sorted(_config_raw.keys()):
        val_str = str(_config_raw[key])
        if len(val_str) > 15:
            val_str = val_str[:12] + "..."
        desc = _PARAM_DESC.get(key, "")
        print(f"  {key:<22} {val_str:<15} {desc}")
else:
    print("No config stored in this result file (re-run the pipeline to get one).")

### Get some statistics on NCC score and area distribution

In [ ]:
# ── Dataset statistics ─────────────────────────────────────────────────────────
_nccs      = [a["ncc"] for a, _ in pairs]
_all_areas = [int(s["mask"].sum()) for s in spots.values()]
_all_nccs  = [s["ncc"] for s in spots.values()]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# ── NCC score histogram ────────────────────────────────────────────────────────
ax = axes[0]
ax.hist(_nccs, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlabel('NCC score')
ax.set_ylabel('Pairs')
ax.set_title('NCC score distribution')

# ── Spot area distribution (log-spaced bins) ───────────────────────────────────
ax = axes[1]
_log_bins = np.logspace(np.log10(max(1, min(_all_areas))),
                        np.log10(max(_all_areas)), 40)
ax.hist(_all_areas, bins=_log_bins, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xscale('log')
ax.set_xlabel('Spot area (px)')
ax.set_ylabel('Spots')
ax.set_title('Spot area distribution  (both partners)')

# ── Area vs NCC scatter ────────────────────────────────────────────────────────
ax = axes[2]
ax.scatter(_all_areas, _all_nccs, s=6, alpha=0.8, color='steelblue', linewidths=2)
ax.set_xscale('log')
ax.set_xlabel('Spot area (px)')
ax.set_ylabel('NCC score')
ax.set_title('Area vs NCC score  (both partners)')

fig.suptitle(
    f"{len(pairs)} Friedel pairs - {len(_all_areas)} spots",
    fontsize=13,
)
plt.tight_layout()
plt.show()

### Define functions for plotting the results

In [ ]:
def _norm(arr):
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.zeros_like(arr) if hi == lo else np.clip((arr - lo) / (hi - lo), 0, 1)

def _com_in_mask(mask):
    ys, xs = np.where(mask)
    return float(ys.mean()), float(xs.mean())

def _overlay(mask_a, mask_b_flipped):
    ha, wa = mask_a.shape
    hb, wb = mask_b_flipped.shape
    ch, cw = max(ha, hb) * 2 + 4, max(wa, wb) * 2 + 4
    cy, cx = ch // 2, cw // 2
    def place(mask, com):
        h, w = mask.shape
        y0, x0 = cy - int(round(com[0])), cx - int(round(com[1]))
        canvas = np.zeros((ch, cw), dtype=bool)
        sy, ey = max(0, y0), min(ch, y0 + h)
        sx, ex = max(0, x0), min(cw, x0 + w)
        canvas[sy:ey, sx:ex] = mask[max(0,-y0):h-max(0,y0+h-ch), max(0,-x0):w-max(0,x0+w-cw)]
        return canvas
    ca = place(mask_a, _com_in_mask(mask_a))
    cb = place(mask_b_flipped, _com_in_mask(mask_b_flipped))
    rgba = np.ones((ch, cw, 4), dtype=np.float32)
    rgba[ca & ~cb] = [1, 0, 0, 1]
    rgba[cb & ~ca] = [0, 0, 1, 1]
    rgba[ca & cb]  = [1, 0, 1, 1]
    return rgba, ch, cw

def _padded_crop(frame, spot):
    mask = spot["mask"]
    ys, xs = np.where(mask)
    com_r, com_c = ys.mean(), xs.mean()
    min_r = round(spot["centroid"][1] - com_r)
    min_c = round(spot["centroid"][0] - com_c)
    h, w = mask.shape
    pr0 = max(0, min_r - CONTEXT)
    pc0 = max(0, min_c - CONTEXT)
    pr1 = min_r + h + CONTEXT
    pc1 = min_c + w + CONTEXT
    pm = np.zeros((pr1 - pr0, pc1 - pc0), dtype=bool)
    pm[min_r - pr0 : min_r - pr0 + h, min_c - pc0 : min_c - pc0 + w] = mask
    crop = frame[pr0:pr1, pc0:pc1] if frame is not None else None
    return crop, pm

def _draw_spot_on_ax(ax, frame, spot, color, hline_y=None):
    if frame is not None:
        ax.imshow(_norm(frame), cmap='gray', interpolation='bilinear')
        spot_cx, spot_cy = spot['centroid']
        ax.plot(spot_cx, spot_cy, '+', color=color, markersize=16, markeredgewidth=1.5)
        if hline_y is not None:
            ax.axhline(hline_y, color='yellow', linestyle=':', linewidth=1, alpha=0.8)

def _add_search_band(ax, frame, y_center, half_width):
    if frame is None:
        return
    h = frame.shape[0]
    y_lo = max(0,     y_center - half_width)
    y_hi = min(h - 1, y_center + half_width)
    ax.axhspan(y_lo, y_hi, color='cyan', alpha=0.25)
    ax.set_ylim(h - 0.5, -0.5)

def _add_x_band(ax, frame, x_center, half_width):
    if frame is None:
        return
    w = frame.shape[1]
    x_lo = max(0,     x_center - half_width)
    x_hi = min(w - 1, x_center + half_width)
    ax.axvspan(x_lo, x_hi, color='orange', alpha=0.25)

def _annotate_raw_ax(ax, frame, spot, color, hline_y, y_expected, x_expected, y_half, x_half):
    _draw_spot_on_ax(ax, frame, spot, color, hline_y=hline_y)
    _add_search_band(ax, frame, y_expected, y_half)
    _add_x_band(ax, frame, x_expected, x_half)

def show_pair(idx):
    a, b = pairs[idx]
    ov, ov_ch, ov_cw = _overlay(a['mask'], b['mask'][:, ::-1])

    raw_a  = _load_raw_frame(a['frame_idx'])
    raw_b  = _load_raw_frame(b['frame_idx'])
    seg_a  = _load_frame(a['frame_idx'])
    seg_b  = _load_frame(b['frame_idx'])

    raw_crop_a, pmask_a = _padded_crop(raw_a, a)
    raw_crop_b, pmask_b = _padded_crop(raw_b, b)
    seg_crop_a, _       = _padded_crop(seg_a, a)
    seg_crop_b, _       = _padded_crop(seg_b, b)

    midline_y    = (a['centroid'][1] + b['centroid'][1]) / 2
    y_expected_b = 2 * BEAM_CENTER_Y - a['centroid'][1]
    search_half  = 2 * BEAM_HALF_HEIGHT

    x_expected_b  = a['centroid'][0]
    x_search_half = 2 * BEAM_HALF_WIDTH

    fig, axes = plt.subplots(3, 4, figsize=(17, 12))
    zoom = dict(interpolation='nearest')

    # ── Row 0: Spot A ──────────────────────────────────────────────────────────
    if raw_crop_a is not None:
        axes[0, 0].imshow(_norm(raw_crop_a), cmap='gray', **zoom)
    axes[0, 0].set_title(f"A - Raw (Frame {a['frame_idx']})")

    if seg_crop_a is not None:
        axes[0, 1].imshow(_norm(seg_crop_a), cmap='gray', **zoom)
    axes[0, 1].set_title(f"A - {SEG_TYPE} (Frame {a['frame_idx']})")

    axes[0, 2].imshow(pmask_a, cmap='hot', **zoom)
    axes[0, 2].plot(*_com_in_mask(pmask_a)[::-1], '+', color='cyan', markersize=12, markeredgewidth=1.5)
    axes[0, 2].set_title(
        f"A - Binary mask (Frame {a['frame_idx']})\n"
        f"Center of Mass: ({a['centroid'][0]:.1f}, {a['centroid'][1]:.1f})\n"
        f"BBox: {a['mask'].shape[0]} x {a['mask'].shape[1]}   Area: {a['mask'].sum()}"
    )

    axes[0, 3].imshow(ov, **zoom)
    axes[0, 3].plot(ov_cw // 2, ov_ch // 2, '+', color='white', markersize=12, markeredgewidth=1.5)
    axes[0, 3].set_title('Center of Mass Overlay\nRed = A ; Blue = B ; Magenta = Overlap\nB flipped vertically')

    # ── Row 1: Spot B (Friedel partner) ────────────────────────────────────────
    if raw_crop_b is not None:
        axes[1, 0].imshow(_norm(raw_crop_b), cmap='gray', **zoom)
    axes[1, 0].set_title(f"B - Raw (Frame {b['frame_idx']})")

    if seg_crop_b is not None:
        axes[1, 1].imshow(_norm(seg_crop_b), cmap='gray', **zoom)
    axes[1, 1].set_title(f"B - {SEG_TYPE} (Frame {b['frame_idx']})")

    axes[1, 2].imshow(pmask_b, cmap='hot', **zoom)
    axes[1, 2].plot(*_com_in_mask(pmask_b)[::-1], '+', color='cyan', markersize=12, markeredgewidth=1.5)
    axes[1, 2].set_title(
        f"B - Binary Mask (Frame {b['frame_idx']})\n"
        f"Center of Mass: ({b['centroid'][0]:.1f}, {b['centroid'][1]:.1f})\n"
        f"BBox: {b['mask'].shape[0]} x {b['mask'].shape[1]}   Area: {b['mask'].sum()}"
    )

    axes[1, 3].text(
        0.5, 0.5,
        f"Pair {idx + 1} / {len(pairs)}\n\nNCC = {a['ncc']:.3f}\n\u0394frame = {abs(a['frame_idx'] - b['frame_idx'])}",
        ha='center', va='center', fontsize=14, transform=axes[1, 3].transAxes,
    )

    # ── Row 2: full frames ─────────────────────────────────────────────────────
    seg_a_bin = (seg_a >= 1) if seg_a is not None else None
    if seg_a_bin is not None:
        axes[2, 0].imshow(seg_a_bin, cmap='gray', vmin=0, vmax=1, interpolation='bilinear')
        axes[2, 0].plot(a['centroid'][0], a['centroid'][1], '+', color='red', markersize=16, markeredgewidth=1.5)
    axes[2, 0].set_title(f"Position of A - Binary mask (Frame {a['frame_idx']})")

    seg_b_bin = (seg_b >= 1) if seg_b is not None else None
    if seg_b_bin is not None:
        axes[2, 1].imshow(seg_b_bin, cmap='gray', vmin=0, vmax=1, interpolation='bilinear')
        axes[2, 1].plot(b['centroid'][0], b['centroid'][1], '+', color='cyan', markersize=16, markeredgewidth=1.5)
    axes[2, 1].set_title(f"Position of B - Binary mask (Frame {b['frame_idx']})")

    _annotate_raw_ax(axes[2, 2], raw_a, a, 'red', BEAM_CENTER_Y,
                     y_expected_b, x_expected_b, search_half, x_search_half)
    axes[2, 2].set_title(
        f"Position of A - Raw (Frame {a['frame_idx']})\n"
        f"Beam Centre Y = {BEAM_CENTER_Y:.1f}\n"
        f"Cyan = Valid Y  Orange = Valid X"
    )

    _annotate_raw_ax(axes[2, 3], raw_b, b, 'cyan', midline_y,
                     y_expected_b, x_expected_b, search_half, x_search_half)
    axes[2, 3].set_title(
        f"Position of B - Raw (Frame {b['frame_idx']})\n"
        f"Grain Y = {midline_y:.1f}\n"
        f"Cyan = Valid Y  Orange = Valid X"
    )

    if all(v is not None for v in [BEAM_CENTER_X, BEAM_CENTER_Y, BEAM_HALF_WIDTH, BEAM_HALF_HEIGHT]):
        _beam_rect = dict(linewidth=1.5, edgecolor='lime', facecolor='none', linestyle='--')
        for _ax in [axes[2, 2], axes[2, 3]]:
            _ax.add_patch(mpatches.Rectangle(
                (BEAM_CENTER_X - BEAM_HALF_WIDTH, BEAM_CENTER_Y - BEAM_HALF_HEIGHT),
                2 * BEAM_HALF_WIDTH, 2 * BEAM_HALF_HEIGHT,
                **_beam_rect,
            ))

    for ax in axes.flat:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

### Plot the results with a slider widget
Original spot = **A** <br>
Partner spot = **B** <br>

The first row shows Spot **A** in 3 stages: the original *raw* image, the *preprocessed* image, the *binary* mask, and an overlay of the binary mask with its most likely matching partner spot.

The second row shows Spot **B**: the *raw* image, *preprocessed* image, and *binary* mask. The final panel is reserved for additional metadata.

The third row illustrates the spatial relationship between Spots **A** and **B**. The first panel shows their positions on the preprocessed image. The second panel displays the raw image of Spot **A** with a cyan band indicating the *estimated* location of Spot **B**. This region is determined by mirroring Spot **A** across the beam centerline and using the beam height as the search range. The final panel shows the raw image of Spot **B** within the predicted region. Here, the centerline is defined as the midpoint between Spots **A** and **B**, indicating the grain position within the specimen relative to the beam (The original y position of the grain in the material).

In [ ]:
slider = widgets.IntSlider(
    value=0, min=0, max=max(len(pairs) - 1, 0), step=1,
    description='Change spot:', continuous_update=False, layout=widgets.Layout(width='99%'),
)
out = widgets.interactive_output(show_pair, {'idx': slider})
display(slider, out)